In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from ultralytics import YOLO

In [ ]:
print("Loading YOLO model...")
model = YOLO('runs/classify/train3/weights/best.pt')
print("Done.")

In [ ]:
print("Loading MediaPipe hand detector...")
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3)
print("Done.")

In [ ]:
def predict_from_video(img):
    cv2.imwrite('saved_image.jpg', cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    results = model('saved_image.jpg')
    prediction = results[0].probs.top1
    return prediction


def box_maker(mhl, h, w):
    x_land = []
    y_land = []
    for hand_landmarks in mhl:
        for i in range(len(hand_landmarks.landmark)):
            x = hand_landmarks.landmark[i].x
            y = hand_landmarks.landmark[i].y
            x_land.append(x)
            y_land.append(y)

        x1 = int(min(x_land)*w)
        y1 = int(min(y_land)*h)
        x2 = int(max(x_land)*w)
        y2 = int(max(y_land)*h)

        width = x2 - x1
        height = y2 - y1
        longer_side = max(width, height)
        new_x1 = x1 + (width - longer_side) // 2 - 20
        new_y1 = y1 + (height - longer_side) // 2 - 20
        new_x2 = new_x1 + longer_side + 20
        new_y2 = new_y1 + longer_side + 20
        new_x1 = abs(new_x1)
        new_x2 = abs(new_x2)
        new_y1 = abs(new_y1)
        new_y2 = abs(new_y2)

    return x1, y1, x2, y2, new_x1, new_y1, new_x2, new_y2


# Change WEBCAM_INDEX to 1 if you have an external webcam
WEBCAM_INDEX = 0

print(f"Opening webcam (index {WEBCAM_INDEX})...")
capture = cv2.VideoCapture(WEBCAM_INDEX)

if not capture.isOpened():
    raise RuntimeError(f"Could not open webcam at index {WEBCAM_INDEX}. Try changing WEBCAM_INDEX above.")

labels_dict = {0: 'alapadma', 1: 'arala', 12: 'ardhachandra', 23: 'ardhapataka', 24: 'bhramhara', 25: 'chandrakala', 26: 'chatura', 27:'hamsapaksha', 28:'hamsasya', 29:'kangula', 2:'kapitha', 3:'kartarimukha', 4:'katakamukha-1', 5:'katakamukha-2', 6:'katakamukha-3', 7:'mayura', 8:'mrighashisha', 9:'mukula', 10:'mushthi', 11:'padmakosha', 13:'pataka', 14:'santamsha', 15:'sarpashisha', 16:'shikhara', 17:'shukatunda', 18:'singhamukha', 19:'suchi', 20:'tamrachuda', 21:'tripataka', 22:'trishula'}

cv2.namedWindow('frame', cv2.WINDOW_AUTOSIZE)
cv2.moveWindow('frame', 100, 100)

print("Ready! A window should appear — check your Dock if you don't see it.")
print("Hold up a hand gesture to see predictions. Press Esc to quit.")

while True:
    ret, frame = capture.read()
    if not ret:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w, _ = rgb.shape
    results = hands.process(rgb)
    if results.multi_hand_landmarks:
        if len(results.multi_hand_landmarks) == 1:
            x1, y1, x2, y2, new_x1, new_y1, new_x2, new_y2 = box_maker(results.multi_hand_landmarks, h, w)
            cropped_frame = rgb[new_y1:new_y2, new_x1:new_x2]

            prediction = predict_from_video(cropped_frame)
            predicted_character = labels_dict[prediction]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), 4)
            cv2.putText(frame, predicted_character, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 2, (0, 0, 0), 3)

        elif len(results.multi_hand_landmarks) == 2:
            x11, y11, x21, y21, new_x11, new_y11, new_x21, new_y21 = box_maker([results.multi_hand_landmarks[0]], h, w)
            x12, y12, x22, y22, new_x12, new_y12, new_x22, new_y22 = box_maker([results.multi_hand_landmarks[1]], h, w)

            cropped_frame1 = rgb[new_y11:new_y21, new_x11:new_x21]
            cropped_frame2 = rgb[new_y12:new_y22, new_x12:new_x22]

            prediction1 = predict_from_video(cropped_frame1)
            prediction2 = predict_from_video(cropped_frame2)

            predicted_character1 = labels_dict[prediction1]
            predicted_character2 = labels_dict[prediction2]

            cv2.rectangle(frame, (x11, y11), (x21, y21), (0, 0, 0), 4)
            cv2.rectangle(frame, (x12, y12), (x22, y22), (0, 0, 0), 4)

            cv2.putText(frame, predicted_character1, (x11, y11 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 3)
            cv2.putText(frame, predicted_character2, (x12, y12 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 3)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) == 27:
        break

capture.release()
cv2.destroyAllWindows()